### Pipeline to generate compressed datasets using pre-trained autoencoders
================================================================================

In [ ]:
import os
import re
import torch
import torch.nn as nn
import netCDF4
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
from pathlib import Path
import datetime
import random
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import multiprocessing
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Desired number of training and testing samples
N_TRAIN = 10000000
N_TEST = 2000000

In [ ]:
class DDMProcessor:
    """
    Class for processing and compressing DDM (Delay Doppler Map) files from NetCDF format.
    """
    @staticmethod
    def check_integrity(f):
        """Check integrity of the netCDF file"""
        if not isinstance(f, netCDF4.Dataset):
            raise ValueError("Input must be a netCDF4.Dataset object")
        if 'raw_counts' not in f.variables:
            raise KeyError("The netCDF file does not contain 'raw_counts' variable")
        if 'sp_alt' not in f.variables or 'sp_inc_angle' not in f.variables:
            raise KeyError("The netCDF file does not contain 'sp_alt' or 'sp_inc_angle' variables")
        if 'sp_rx_gain_copol' not in f.variables or 'sp_rx_gain_xpol' not in f.variables or 'ddm_snr' not in f.variables:
            raise KeyError("The netCDF file does not contain 'sp_rx_gain_copol', 'sp_rx_gain_xpol' or 'ddm_snr' variables")
        if 'sp_lat' not in f.variables or 'sp_lon' not in f.variables:
            raise KeyError("The netCDF file does not contain 'sp_lat' or 'sp_lon' variables")
        if 'sp_surface_type' not in f.variables:
            raise KeyError("The netCDF file does not contain 'sp_surface_type' variable")
        if 'ac_alt' not in f.variables:
            raise KeyError("The netCDF file does not contain 'ac_alt' variable")
        if f.variables['raw_counts'].ndim != 4:
            raise ValueError("The 'raw_counts' variable must have 4 dimensions")
        
    def __init__(self, input_folder, output_folder, device=None):
        """
        Initialize the DDM Processor.
        
        Args:
            input_folder (str): Path to folder containing input NetCDF files
            output_folder (str): Path to folder where compressed files will be saved
            device (torch.device): Device for computation (cuda/cpu)
        """
        self.input_folder = Path(input_folder)
        self.output_folder = Path(output_folder)
        
        # Create output folder if it doesn't exist
        self.output_folder.mkdir(parents=True, exist_ok=True)
        
        # Set device
        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = device
            
        print(f"Using device: {self.device}")
        
        # Initialize scaler
        self.scaler = MinMaxScaler()
        self.scaler_fitted = False
        
        # Placeholder for encoder model (to be loaded or set)
        self.encoder = None
        
    def set_encoder(self, encoder_model):
        """
        Set the encoder model for compression.
        
        Args:
            encoder_model: PyTorch model for encoding/compression
        """
        self.encoder = encoder_model.to(self.device)
        self.encoder.eval()
        
    def load_encoder(self, model_path):
        """
        Load a pre-trained encoder model.
        
        Args:
            model_path (str): Path to the saved encoder model
        """
        # Example implementation - adjust based on your model architecture
        self.encoder = torch.load(model_path, map_location=self.device)
        self.encoder.eval()
        
    def preprocess_snr_filtered(self, f):
        """ Preprocess the netCDF file and return fit data, labels and specular point centers with filtering on signal-to-noise ratio """
        # Check integrity of the netCDF file
        self.check_integrity(f)

        raw_counts = f.variables['raw_counts'][:]
        ac_alt = f.variables['ac_alt'][:]
        sp_alt = f.variables['sp_alt'][:]
        copol = f.variables['sp_rx_gain_copol'][:]
        xpol = f.variables['sp_rx_gain_xpol'][:]
        snr = f.variables['ddm_snr'][:]
        sp_inc_angle = f.variables['sp_inc_angle'][:]
        sp_center_bin_delay_row = f.variables['brcs_ddm_peak_bin_delay_row'][:]
        sp_center_bin_delay_col = f.variables['brcs_ddm_peak_bin_dopp_col'][:]

        distance_2d = (ac_alt[:, np.newaxis] - sp_alt) / np.cos(np.deg2rad(sp_inc_angle)) # Distance between the aircraft and the specular point

        # Filtering mask
        keep_mask = (
            (copol >= 5) & # # SP copolarized gain
            (xpol >= 5) & # SP cross-polarized gain
            (snr > 0) & # Positive signal-to-Noise Ratio
            (distance_2d >= 2000) & #SP distance min
            (distance_2d <= 10000) & #SP distance max
            ~np.isnan(copol) & 
            ~np.isnan(xpol) & 
            ~np.isnan(snr) & 
            ~np.isnan(distance_2d)
        )

        aux_row_counts = np.full(raw_counts.shape, np.nan, dtype=np.float32)
        aux_array_sp_row = np.full(sp_center_bin_delay_row.shape, np.nan, dtype=np.float32)
        aux_array_sp_col = np.full(sp_center_bin_delay_col.shape, np.nan, dtype=np.float32)
        
        i_indices, j_indices = np.where(keep_mask)
        aux_row_counts[i_indices, j_indices] = raw_counts[i_indices, j_indices]
        aux_array_sp_row[i_indices, j_indices] = sp_center_bin_delay_row[i_indices, j_indices]
        aux_array_sp_col[i_indices, j_indices] = sp_center_bin_delay_col[i_indices, j_indices]

        assert aux_row_counts.shape[0] == aux_array_sp_row.shape[0] , "First dimension mismatch among aux_row_counts and aux_array_sp_row arrays"
        assert aux_row_counts.shape[0] == aux_array_sp_col.shape[0] , "First dimension mismatch among aux_row_counts and aux_array_sp_col arrays"

        n_time, n_samples = raw_counts.shape[:2]
        aux_raw_counts_reshaped = aux_row_counts.reshape(n_time * n_samples, *raw_counts.shape[2:])
        aux_array_sp_row_reshaped = aux_array_sp_row.reshape(n_time * n_samples, *aux_array_sp_row.shape[2:])
        aux_array_sp_col_reshaped = aux_array_sp_col.reshape(n_time * n_samples, *aux_array_sp_col.shape[2:])

        assert aux_raw_counts_reshaped.shape[0] == aux_array_sp_row_reshaped.shape[0], "First dimension mismatch among aux_raw_counts_reshaped and aux_array_sp_row_reshaped arrays after reshaping"
        assert aux_raw_counts_reshaped.shape[0] == aux_array_sp_col_reshaped.shape[0], "First dimension mismatch among aux_raw_counts_reshaped and aux_array_sp_col_reshaped arrays after reshaping"
        #freeing memory
        del aux_row_counts
        del aux_array_sp_row
        del aux_array_sp_col

        # Filter out NaN and zero-sum rows
        valid_mask = ~np.any(np.isnan(aux_raw_counts_reshaped), axis=(1, 2)) & (np.sum(aux_raw_counts_reshaped, axis=(1, 2)) > 0)

        sp_center_bin_delay_row_filtered = aux_array_sp_row_reshaped[valid_mask]
        sp_center_bin_delay_col_filtered = aux_array_sp_col_reshaped[valid_mask]
        sp_centers_filtered = list(zip(sp_center_bin_delay_row_filtered.flatten(), sp_center_bin_delay_col_filtered.flatten()))

        fit_data = aux_raw_counts_reshaped[valid_mask].reshape(valid_mask.sum(), -1)

        assert len(sp_centers_filtered) == fit_data.shape[0], "First dimension sp_centers_filtered and fit_data mismatch after reshaping"


        surface_types = np.nan_to_num(f.variables["sp_surface_type"][:], nan=0).ravel()
        label_data = np.isin(surface_types, np.arange(1, 8)).astype(np.int32)
        label_data = label_data[valid_mask]
        
        # Ensure that fit_data and label_data have the same length
        assert fit_data.shape[0] == len(label_data), \
            f"Shape mismatch: fit_data {fit_data.shape[0]}, label_data {len(label_data)}"

        assert len(sp_centers_filtered) == fit_data.shape[0], "First dimension sp_centers_filtered and fit_data mismatch after reshaping"

        return fit_data, label_data, sp_centers_filtered
    
    def preprocess_snr_unfiltered(self, f):
        """ Preprocess the netCDF file and return fit data, labels and specular point centers without filtering on signal-to-noise ratio """
        # Check integrity of the netCDF file
        self.check_integrity(f)

        raw_counts = f.variables['raw_counts'][:]
        ac_alt = f.variables['ac_alt'][:]
        sp_alt = f.variables['sp_alt'][:]
        sp_inc_angle = f.variables['sp_inc_angle'][:]
        copol = f.variables['sp_rx_gain_copol'][:]
        xpol = f.variables['sp_rx_gain_xpol'][:]
        #snr = f.variables['ddm_snr'][:]
        sp_center_bin_delay_row = f.variables['brcs_ddm_peak_bin_delay_row'][:]
        sp_center_bin_delay_col = f.variables['brcs_ddm_peak_bin_dopp_col'][:]

        #Distance between the aircraft and the specular point
        distance_2d = (ac_alt[:, np.newaxis] - sp_alt) / np.cos(np.deg2rad(sp_inc_angle))
        # Filtering mask without SNR
        keep_mask = (
            (copol >= 5) & 
            (xpol >= 5) & 
        #   (snr > 0)  &
            (distance_2d >= 2000) & 
            (distance_2d <= 10000) &
            ~np.isnan(copol) & 
            ~np.isnan(xpol) & 
            #~np.isnan(snr) & 
            ~np.isnan(distance_2d)
        )
    
        aux_row_counts = np.full(raw_counts.shape, np.nan, dtype=np.float32)
        aux_array_sp_row = np.full(sp_center_bin_delay_row.shape, np.nan, dtype=np.float32)
        aux_array_sp_col = np.full(sp_center_bin_delay_col.shape, np.nan, dtype=np.float32)

        i_indices, j_indices = np.where(keep_mask)

        aux_row_counts[i_indices, j_indices] = raw_counts[i_indices, j_indices]
        aux_array_sp_row[i_indices, j_indices] = sp_center_bin_delay_row[i_indices, j_indices]
        aux_array_sp_col[i_indices, j_indices] = sp_center_bin_delay_col[i_indices, j_indices]

        assert aux_row_counts.shape[0] == aux_array_sp_row.shape[0] , "First dimension mismatch among aux_row_counts and aux_array_sp_row arrays"
        assert aux_row_counts.shape[0] == aux_array_sp_col.shape[0] , "First dimension mismatch among aux_row_counts and aux_array_sp_col arrays"

        n_time, n_samples = raw_counts.shape[:2]
        aux_raw_counts_reshaped = aux_row_counts.reshape(n_time * n_samples, *raw_counts.shape[2:])
        aux_array_sp_row_reshaped = aux_array_sp_row.reshape(n_time * n_samples, *aux_array_sp_row.shape[2:])
        aux_array_sp_col_reshaped = aux_array_sp_col.reshape(n_time * n_samples, *aux_array_sp_col.shape[2:])


        assert aux_raw_counts_reshaped.shape[0] == aux_array_sp_row_reshaped.shape[0], "First dimension mismatch among aux_raw_counts_reshaped and aux_array_sp_row_reshaped arrays after reshaping"
        assert aux_raw_counts_reshaped.shape[0] == aux_array_sp_col_reshaped.shape[0], "First dimension mismatch among aux_raw_counts_reshaped and aux_array_sp_col_reshaped arrays after reshaping"

        del aux_row_counts
        del aux_array_sp_row
        del aux_array_sp_col

        valid_mask = ~np.any(np.isnan(aux_raw_counts_reshaped), axis=(1, 2)) & (np.sum(aux_raw_counts_reshaped, axis=(1, 2)) > 0)

        sp_center_bin_delay_row_filtered = aux_array_sp_row_reshaped[valid_mask]
        sp_center_bin_delay_col_filtered = aux_array_sp_col_reshaped[valid_mask]
        sp_centers_filtered = list(zip(sp_center_bin_delay_row_filtered.flatten(), sp_center_bin_delay_col_filtered.flatten()))

        fit_data = aux_raw_counts_reshaped[valid_mask].reshape(valid_mask.sum(), -1)

        assert len(sp_centers_filtered) == fit_data.shape[0], "First dimension sp_centers_filtered and fit_data mismatch after reshaping"
    
        surface_types = np.nan_to_num(f.variables["sp_surface_type"][:], nan=0).ravel()
        label_data = np.isin(surface_types, np.arange(1, 8)).astype(np.int32)
        label_data = label_data[valid_mask]

        # Ensure that fit_data and label_data have the same length
        assert fit_data.shape[0] == len(label_data), \
            f"Shape mismatch: fit_data {fit_data.shape[0]}, label_data {len(label_data)}"

        assert len(sp_centers_filtered) == fit_data.shape[0], "First dimension sp_centers_filtered and fit_data mismatch after reshaping"

        return fit_data, label_data, sp_centers_filtered
    

    def normalize_data(self, ddm_data_raw):
        """
        Normalize DDM data to [0, 1] range.
        
        Args:
            ddm_data_raw (np.ndarray): Raw DDM data
            
        Returns:
            np.ndarray: Normalized DDM data
        """
        # Scale the data
        ddm_data = self.scaler.fit_transform(ddm_data_raw * 1e13)
        self.scaler_fitted = True
        
        return ddm_data
    
    def compress_data(self, tensor_data):
        """
        Compress data using the encoder model.
        
        Args:
            tensor_data (torch.Tensor): Input tensor data
            
        Returns:
            np.ndarray: Compressed data
        """
        if self.encoder is None:
            raise ValueError("Encoder model not set. Use set_encoder() or load_encoder() first.")
        
        # Create dataset and dataloader
        dataset = TensorDataset(tensor_data)
        dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
        
        compressed_data = []
        
        # Compress data in batches
        with torch.no_grad():
            for batch in dataloader:
                inputs = batch[0].to(self.device)
                compressed = self.encoder(inputs)
                compressed_data.append(compressed.cpu().numpy())
        
        # Concatenate all compressed batches
        compressed_array = np.concatenate(compressed_data, axis=0)
        
        return compressed_array
    
    def save_compressed_data(self, compressed_data, original_filename):
        """
        Save compressed data to output folder.
        
        Args:
            compressed_data (np.ndarray): Compressed data to save
            original_filename (str): Original filename (without extension)
        """
        output_filename = f"{original_filename}_compressed.npz"
        output_path = self.output_folder / output_filename
        
        # Save compressed data using numpy compressed format
        np.savez_compressed(output_path, data=compressed_data)
        #print(f"  Saved compressed data to {output_path}")
        
    def process_all_files(self, file_extension='.nc', save_scaler=True):
        """
        Process all NetCDF files in the input folder.
        
        Args:
            file_extension (str): Extension of files to process (default: '.nc')
            save_scaler (bool): Whether to save the scaler for future use
        """
        from collections import defaultdict


        # Get all files with specified extension
        file_list = list(self.input_folder.glob(f'*{file_extension}'))# Limit to first 50 files for testing

        
        if len(file_list) == 0:
            print(f"No files with extension '{file_extension}' found in {self.input_folder}")
            return
        
        print(f"Found {len(file_list)} files to process")
        full_data_dict = defaultdict(dict)
        # Process each file
        for file_path in tqdm(file_list, desc="Processing files"):
            if not file_path.is_file():
                continue
            # Step 1: Load and process DDM data
            try:
                f = netCDF4.Dataset(f'{file_path}', 'r')
                ddm_data_raw, label_data, _ = self.preprocess_snr_unfiltered(f) # type: ignore
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")
                continue
            #full_data_dict[data_dict['file_name']] = data_dict
            if ddm_data_raw is None:
                continue
            if label_data is None:
                continue
            
            # Step 2: Normalize data
            ddm_data_normalized = self.normalize_data(ddm_data_raw)
            
            # Step 3: Convert to tensor
            tensor_data = torch.tensor(ddm_data_normalized, dtype=torch.float32)
            
            # Step 4: Compress data (if encoder is available)
            if self.encoder is not None:
                compressed_data = self.compress_data(tensor_data)
                
                # Step 5: Save compressed data
                filename_without_ext = file_path.stem
                self.save_compressed_data(compressed_data, filename_without_ext)
                #print(f"  Saving normalized data to {filename_without_ext}_normalized.npz")

                full_data_dict[str(filename_without_ext)]['compressed_data'] = compressed_data # type: ignore
                full_data_dict[str(filename_without_ext)]['labels'] = label_data  # type: ignore


            else:
                # If no encoder, save normalized data
                print("  No encoder set - saving normalized data instead")
                filename_without_ext = file_path.stem
               
                output_filename = f"{filename_without_ext}_normalized.npz"
                output_path = self.output_folder / output_filename
                np.savez_compressed(output_path, data=ddm_data_normalized)
                print(f"  Saved normalized data to {output_path}")
            
            
        
        # Save scaler for future use
        if save_scaler and self.scaler_fitted:
            scaler_path = self.output_folder / "scaler_encoder.pkl"
            joblib.dump(self.scaler, scaler_path)
            print(f"\nScaler saved to {scaler_path}")
        
        print(f"\n{'='*50}")
        print(f"Processing complete! Output files saved to {self.output_folder}")
        return full_data_dict # type: ignore

In [9]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(200, 100),
            nn.ReLU(),
            nn.Linear(100, 20),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 100),
            nn.ReLU(),
            nn.Linear(100, 200)
        )

    def forward(self, x):
        return self.net(x)

# ----------------------
# Load the saved models
# ----------------------
def load_model(model_class, path):
    model = model_class().to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model


In [ ]:
# Old encoder usage

input_folder = "E:/data/RONGOWAI_L1_SDR_V1.0"
output_folder = "E:/data/geo_k_compressed_raw_counts_encoder_old"


processor = DDMProcessor(input_folder, output_folder)


encoder = load_model(Encoder, "C:\\Users\\atogni\\Desktop\\rongowai\\geo_k\\raw_counts\\encoder_all_surface2.pth")
processor.set_encoder(encoder)

full_data_dict_old_encoder = processor.process_all_files(file_extension='.nc', save_scaler=True)

In [ ]:
# New encoder usage
input_folder = "E:/data/RONGOWAI_L1_SDR_V1.0"
output_folder = "E:/data/geo_k_compressed_raw_counts_enh"

processor = DDMProcessor(input_folder, output_folder)

encoder = load_model(Encoder, "C:/Users/atogni/Desktop/rongowai/geo_k/raw_counts/encoder_enh.pth")
processor.set_encoder(encoder)

full_data_dict_enh_encoder = processor.process_all_files(file_extension='.nc', save_scaler=True)

Using device: cuda
Found 4942 files to process


Processing files:  50%|█████     | 2475/4942 [22:58<23:21,  1.76it/s]  

Error processing file E:\data\RONGOWAI_L1_SDR_V1.0\20231104-094532_NZWB-NZAA_L1.nc: [Errno -101] NetCDF: HDF error: 'E:\\data\\RONGOWAI_L1_SDR_V1.0\\20231104-094532_NZWB-NZAA_L1.nc'


Processing files:  50%|█████     | 2488/4942 [23:06<21:55,  1.86it/s]

Error processing file E:\data\RONGOWAI_L1_SDR_V1.0\20231107-134644_NZAA-NZKK_L1.nc: [Errno -101] NetCDF: HDF error: 'E:\\data\\RONGOWAI_L1_SDR_V1.0\\20231107-134644_NZAA-NZKK_L1.nc'


...

Error processing file E:\data\RONGOWAI_L1_SDR_V1.0\20250213-071529_NZAA-NZNP_L1.nc: cannot reshape array of size 0 into shape (0,newaxis)
Error processing file E:\data\RONGOWAI_L1_SDR_V1.0\20250213-082601_NZNP-NZAA_L1.nc: cannot reshape array of size 0 into shape (0,newaxis)


Processing files:  96%|█████████▋| 4764/4942 [45:17<00:39,  4.54it/s]

Error processing file E:\data\RONGOWAI_L1_SDR_V1.0\20250213-101229_NZAA-NZNP_L1.nc: cannot reshape array of size 0 into shape (0,newaxis)
Error processing file E:\data\RONGOWAI_L1_SDR_V1.0\20250213-115907_NZNP-NZAA_L1.nc: cannot reshape array of size 0 into shape (0,newaxis)


Processing files: 100%|██████████| 4942/4942 [46:54<00:00,  1.76it/s]


Scaler saved to E:\data\geo_k_compressed_raw_counts_enh\scaler_encoder.pkl

Processing complete! Output files saved to E:\data\geo_k_compressed_raw_counts_enh


## Save the processed data

In [ ]:
dfs = []
for key in full_data_dict_old_encoder.keys():
    data = full_data_dict_old_encoder[key]
    try:
        df = pd.DataFrame(data["compressed_data"])
        df["label"] = data["labels"]
        dfs.append(df)
    except Exception as e:
        print(f"Error processing {key}: {e}")       

merged_df_old_encoder = pd.concat(dfs, ignore_index=True)
#del full_data_dict_old_encoder
del dfs

In [13]:
dfs = []
for key in full_data_dict_enh_encoder.keys():
    data = full_data_dict_enh_encoder[key]
    try:
        df = pd.DataFrame(data["compressed_data"])
        df["label"] = data["labels"]
        dfs.append(df)
    except Exception as e:
        print(f"Error processing {key}: {e}")       

merged_df_enh_encoder = pd.concat(dfs, ignore_index=True)
del full_data_dict_enh_encoder
del dfs

In [14]:
def create_balanced_train_test_split(df, label_column, n_train, n_test, random_state=42):
    """
    Crea un dataset di training bilanciato con N righe e un dataset di test bilanciato con M righe,
    senza sovrapposizioni tra i due dataset.
   
    Parameters:
    -----------
    df : pandas.DataFrame
        Il dataset originale
    label_column : str
        Nome della colonna contenente le labels
    n_train : int
        Numero di righe per il dataset di training
    n_test : int
        Numero di righe per il dataset di test
    random_state : int, default=42
        Seed per la riproducibilità
       
    Returns:
    --------
    tuple : (train_df, test_df)
        Tupla contenente il dataset di training bilanciato e il dataset di test bilanciato
    """
    import pandas as pd
    import numpy as np
   
    # Verifica che ci siano abbastanza righe in totale
    if len(df) < n_train + n_test:
        raise ValueError(f"Dataset troppo piccolo: {len(df)} righe disponibili, "
                        f"ma richieste {n_train + n_test} righe totali")
   
    # Crea una copia per evitare modifiche al dataset originale
    df_copy = df.copy()
    
    # Ottieni le label uniche e i loro conteggi
    label_counts = df_copy[label_column].value_counts()
    unique_labels = label_counts.index.tolist()
    n_labels = len(unique_labels)
    
    # Calcola quante righe per label per training e test
    n_per_label_train = n_train // n_labels
    n_per_label_test = n_test // n_labels
    
    # Verifica che ci siano abbastanza esempi per ogni label
    min_samples_per_label = label_counts.min()
    required_per_label = n_per_label_train + n_per_label_test
    
    if min_samples_per_label < required_per_label:
        raise ValueError(f"Label '{label_counts.idxmin()}' ha solo {min_samples_per_label} esempi, "
                        f"ma ne servono {required_per_label} ({n_per_label_train} train + {n_per_label_test} test)")
    
    # Imposta il seed per la riproducibilità
    np.random.seed(random_state)
    
    train_samples = []
    test_samples = []
    
    for label in unique_labels:
        label_df = df_copy[df_copy[label_column] == label]
        
        # Mescola i dati per questa label
        label_df_shuffled = label_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
        
        # Prendi i primi n_per_label_train per il training
        train_sample = label_df_shuffled.iloc[:n_per_label_train]
        train_samples.append(train_sample)
        
        # Prendi i successivi n_per_label_test per il test
        test_sample = label_df_shuffled.iloc[n_per_label_train:n_per_label_train + n_per_label_test]
        test_samples.append(test_sample)
    
    # Combina tutti i campioni
    train_df = pd.concat(train_samples, ignore_index=True)
    test_df = pd.concat(test_samples, ignore_index=True)
    
    # Mescola l'ordine finale dei dataset
    train_df = train_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # Verifica finale delle dimensioni
    print(f"Training set creato: {len(train_df)} righe")
    print(f"Test set creato: {len(test_df)} righe")
    print("\nDistribuzione training set:")
    print(train_df[label_column].value_counts().sort_index())
    print("\nDistribuzione test set:")
    print(test_df[label_column].value_counts().sort_index())
    
    return train_df, test_df

In [ ]:
train_df_old_encoder, test_df_old_encoder = create_balanced_train_test_split(merged_df_old_encoder, label_column="label", n_train=N_TRAIN, n_test=N_TEST)

table_ = pa.Table.from_pandas(train_df_old_encoder, preserve_index=False)
pq.write_table(
    table_,
    'E:/data/balanced_df_old_encoder_10M.parquet',
    compression='zstd',
    use_dictionary=True,
)
table_ = pa.Table.from_pandas(test_df_old_encoder, preserve_index=False)
pq.write_table(
    table_,
    'E:/data/test_df_old_encoder_2M.parquet',
    compression='zstd',
    use_dictionary=True,
)

In [ ]:
del table_, test_df_old_encoder, train_df_old_encoder, merged_df_old_encoder

In [16]:
train_df_enh_encoder, test_df_enh_encoder = create_balanced_train_test_split(merged_df_enh_encoder, label_column="label", n_train=N_TRAIN, n_test=N_TEST)
table_ = pa.Table.from_pandas(train_df_enh_encoder, preserve_index=False)
pq.write_table(
    table_,
    'E:/data/balanced_df_enh_encoder_10M.parquet',
    compression='zstd',
    use_dictionary=True,
)

table_ = pa.Table.from_pandas(test_df_enh_encoder, preserve_index=False)
pq.write_table(
    table_,
    'E:/data/test_df_enh_encoder_2M.parquet',
    compression='zstd',
    use_dictionary=True,
)


Training set creato: 10000000 righe
Test set creato: 2000000 righe

Distribuzione training set:
label
0    5000000
1    5000000
Name: count, dtype: int64

Distribuzione test set:
label
0    1000000
1    1000000
Name: count, dtype: int64


In [17]:
del table_, test_df_enh_encoder, train_df_enh_encoder, merged_df_enh_encoder